In [14]:
# RabbitMQ Complete Guide - From Scratch

## What is RabbitMQ?

RabbitMQ is an open-source message broker that implements the Advanced Message Queuing Protocol (AMQP). It helps applications communicate asynchronously by sending and receiving messages through queues.

---

## Key Components in RabbitMQ

### 1. **Publisher/Producer**
- Application or service that **sends messages**
- Publishes messages to exchanges
- Example: Order service that sends order notifications

### 2. **Subscriber/Consumer**
- Application or service that **receives messages**
- Consumes messages from queues
- Example: Email service that sends confirmation emails

### 3. **Queue**
- **Buffer that stores messages** until consumed
- Messages wait here until a consumer picks them up
- Can be durable (survives server restart) or temporary

### 4. **Exchange**
- **Routes messages to queues** based on rules
- Acts as a mail exchange/post office
- Different exchange types use different routing rules

### 5. **Binding**
- **Link between exchange and queue** with routing rules
- Defines which messages from an exchange go to which queue
- Uses binding keys (or patterns) to match routing keys

### 6. **Message**
- **Data payload** sent from producer to consumer
- Contains the actual information being transferred
- Can include headers, properties, and metadata

### 7. **Channel**
- **Lightweight virtual connection** created on top of TCP connection
- Multiple channels can share one TCP connection
- Consumes less resources than maintaining multiple TCP connections
- Allows concurrent operations within a single connection

### 8. **Routing Key**
- **String that determines message routing** from exchange to queues
- Producer specifies routing key when publishing
- Exchange uses routing key + binding rules to route messages

---

## Message Flow in RabbitMQ

### Basic Flow:
```
Publisher → Exchange → Binding → Queue → Consumer
```

### Detailed Flow:

1. **Publisher** sends message to **Exchange** with a **Routing Key**
2. **Exchange** receives message and routing key
3. **Exchange** uses **Binding Rules** to determine which queue(s) should receive the message
4. Message is routed to **Queue(s)** based on binding rules
5. **Consumer** pulls message from **Queue**
6. **Consumer** processes message and sends acknowledgment (ACK)

### Visual Flow:
```
Publisher 
    ↓ (message + routing_key)
Exchange 
    ↓ (binding rules match routing_key)
Binding 
    ↓ (routes message)
Queue 
    ↓ (stores message)
Consumer 
    ↓ (processes and ACKs)
```

### Example Flow:
```
Order Service (Publisher)
    ↓ publishes: {"order_id": "123", "status": "created"} with routing_key="order.created"
Orders Exchange
    ↓ binding rule: routing_key="order.created" → order_created_queue
Order Queue
    ↓ message waiting
Email Service (Consumer)
    ↓ consumes and sends confirmation email
```

---

## Types of Exchanges

RabbitMQ has **4 types of exchanges**, each with different routing behavior:

### 1. **Direct Exchange**
- **Routing Key must match Binding Key exactly**
- Used for: Exact matching scenarios
- Example: Logging system
  - `error` routing key → `error_logs_queue`
  - `warning` routing key → `warning_logs_queue`
  - `info` routing key → `info_logs_queue`

**When to use:** When you need precise, one-to-one message routing

---

### 2. **Topic Exchange**
- **Similar to Direct but supports wildcard patterns**
- Wildcards:
  - `*` (star) = matches **exactly one word**
  - `#` (hash) = matches **zero or more words**
- Used for: Pattern-based routing
- Example: Order processing system
  - `order.payment.success` → matches `order.*.success`
  - `order.shipping.failed` → matches `order.*.failed`
  - `user.login` → matches `user.*`

**When to use:** When you need flexible, pattern-based message routing

---

### 3. **Fanout Exchange**
- **Ignores routing keys entirely**
- **Broadcasts to ALL bound queues**
- Each queue gets its own copy of the message
- Used for: Broadcasting notifications to multiple services
- Example: E-commerce order placement
  - One order placed → notifies:
    - Inventory service queue
    - Email service queue
    - SMS service queue
    - Shipping service queue

**When to use:** When you need to send the same message to multiple consumers simultaneously

**Visual:**
```
Publisher sends 1 message
    ↓
Fanout Exchange
    ↓
┌────┴────┐
↓    ↓    ↓
Queue1 Queue2 Queue3  (Each queue gets its OWN COPY)
↓    ↓    ↓
Worker1 Worker2 Worker3  (Each processes independently)
```

---

### 4. **Headers Exchange**
- **Routes based on message headers** (not routing keys)
- Routing key is **ignored**
- Uses `x-match` argument:
  - `x-match: 'all'` → ALL specified headers must match
  - `x-match: 'any'` → ANY of the specified headers must match
- Used for: Complex routing based on message metadata
- Example: Priority-based routing
  - Headers: `{'priority': 'high', 'type': 'critical'}` 
  - Matches queue with: `x-match: 'all', priority: 'high', type: 'critical'`

**When to use:** When routing logic is too complex for routing keys or when you need to route based on message properties

---

## Routing Key vs Binding Key

### Routing Key
- **Set by Publisher** when sending message
- Determines how message is routed
- Example: `channel.basic_publish(routing_key='error', ...)`

### Binding Key
- **Set when binding queue to exchange**
- Defines the rule for which messages queue should receive
- For Direct: Must match routing key exactly
- For Topic: Can use wildcards to match routing key patterns
- Example: `channel.queue_bind(routing_key='error')` or `routing_key='order.*.success'`

### Matching Logic:
- **Direct Exchange:** `routing_key == binding_key` (exact match)
- **Topic Exchange:** `routing_key` matches `binding_key` pattern (wildcards allowed)
- **Fanout Exchange:** No matching (all queues receive message)
- **Headers Exchange:** Headers match binding arguments (no routing key used)

---

## Real-World Examples by Exchange Type

### Direct Exchange Example:
**Logging System**
```
error messages → error_logs_queue (only)
warning messages → warning_logs_queue (only)
info messages → info_logs_queue (only)
```

### Topic Exchange Example:
**Microservices Communication**
```
order.payment.success → payment_service_queue
order.payment.failed → payment_service_queue
user.created → user_service_queue
user.deleted → user_service_queue
```

### Fanout Exchange Example:
**E-commerce Order Processing**
```
Order Placed Event → Fanout Exchange → 
    ├─ Inventory Service Queue (update stock)
    ├─ Email Service Queue (send confirmation)
    ├─ SMS Service Queue (send SMS)
    └─ Shipping Service Queue (create shipment)
```

### Headers Exchange Example:
**Priority-Based Message Routing**
```
Message with headers {priority: 'high', type: 'urgent'}
    ↓
Routes to: high_priority_queue (all headers match)
```

---

## Connection Architecture

### TCP Connection → Channels
- One **TCP connection** can have multiple **channels**
- Channels are lightweight (less resource-intensive than TCP connections)
- Different channels can be used for different operations concurrently
- Example:
  ```python
  connection = pika.BlockingConnection(...)  # TCP connection
  channel_1 = connection.channel()  # Channel 1
  channel_2 = connection.channel()  # Channel 2 (shares same TCP)
  ```

---

## Summary Table

| Component | Purpose | Key Feature |
|-----------|---------|-------------|
| **Publisher** | Sends messages | Creates messages with routing keys |
| **Exchange** | Routes messages | Different types = different routing rules |
| **Binding** | Links exchange to queue | Defines routing rules |
| **Queue** | Stores messages | Buffer between producer and consumer |
| **Consumer** | Receives messages | Processes messages from queue |
| **Channel** | Virtual connection | Lightweight, multiple per TCP connection |
| **Routing Key** | Message routing identifier | Determines message destination |

---

**Reference Video:** https://www.youtube.com/watch?v=IGuVVElY-DY


SyntaxError: invalid syntax (1545803468.py, line 5)

---

## Quick Navigation

Use this guide to navigate through different RabbitMQ concepts:

1. **Basic Connection** - How to connect and create channels
2. **Direct Exchange** - Exact routing key matching
3. **Topic Exchange** - Pattern-based routing with wildcards
4. **Fanout Exchange** - Broadcast to all queues
5. **Headers Exchange** - Header-based routing
6. **Consumer Implementation** - How to receive messages
7. **Complete Examples** - Real-world use cases
8. **Advanced Features** - ACK, QoS, Durability, DLQ, etc.

---



## Basic Connection Setup

### Installation
```bash
pip install pika

# Start RabbitMQ (macOS)
brew services start rabbitmq

# Or using Docker
docker run -d --name rabbitmq -p 5672:5672 -p 15672:15672 rabbitmq:3-management
```

### Management UI
- URL: http://localhost:15672
- Default credentials: guest / guest

In [11]:
import pika 


connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))

# creating channels (but we can not name them)
channel_1 = connection.channel()
channel_2 = connection.channel()

# Creating queues
channel_1.queue_declare(queue='c1_q1')
channel_2.queue_declare(queue='c2_q2')


# Publishing messages message to 
channel_1.basic_publish(exchange='', routing_key='c1_q1', body='Hello World')


# To close all existing channels, you should call the `close()` method on each channel individually:
channel_1.close()
channel_2.close()

# Optionally, you can also close the connection itself, which will close all channels associated with it:
connection.close()

Direct Exchange
real time example os logging system where i can have my bindig key as log type and accourdengly my system can work 

error messages → only error_logs_queue
warning messages → only warning_logs_queue
info messages → only info_logs_queue


In [12]:
# Direct: Rouating key must match with binding key

import pika 

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))

channel = connection.channel()

channel.exchange_declare(exchange='exchange_1', exchange_type='direct')


channel.queue_declare(queue='queue_1')
channel.queue_declare(queue='queue_2')

channel.queue_bind(exchange='exchange_1', queue='queue_1', routing_key='key_1') # message with routing key 'key_1' will be routed to queue_1
channel.queue_bind(exchange='exchange_1', queue='queue_2', routing_key='key_2') # message with routing key 'key_2' will be routed to queue_2


channel.basic_publish(exchange='exchange_1', routing_key='key_1', body='Hello World') # this will be routed to queue_1
channel.basic_publish(exchange='exchange_1', routing_key='key_2', body='Hello World') # this will be routed to queue_2



error messages → only error_logs_queue
warning messages → only warning_logs_queue
info messages → only info_logs_queue


In [ ]:
# Topic Exchange: Pattern-based routing with wildcards

import pika 

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.exchange_declare(exchange='exchange_2', exchange_type='topic', durable=True)

channel.queue_declare(queue='queue_1', durable=True)
channel.queue_declare(queue='queue_2', durable=True)

# Examples of * (matches exactly ONE word)
channel.queue_bind(exchange='exchange_2', queue='queue_1', routing_key='*.orange.*')  # Matches: quick.orange.rabbit, lazy.orange.elephant
channel.queue_bind(exchange='exchange_2', queue='queue_1', routing_key='*.*.rabbit')  # Matches: quick.orange.rabbit, lazy.pink.rabbit

# Examples of # (matches ZERO or MORE words)
channel.queue_bind(exchange='exchange_2', queue='queue_2', routing_key='lazy.#')      # Matches: lazy, lazy.orange, lazy.orange.elephant
channel.queue_bind(exchange='exchange_2', queue='queue_2', routing_key='#.rabbit')   # Matches: rabbit, quick.rabbit, quick.orange.rabbit

# Key Differences:
# * (star) = matches exactly ONE word
# # (hash) = matches ZERO or MORE words

# Pattern Examples Explained:
# *.orange.*   → Matches: word1.orange.word3 (exactly 3 words, orange in middle)
# #.orange.#   → Matches: orange, word1.orange, orange.word2, word1.orange.word2 (orange anywhere)
# #.orange.*   → Matches: orange.word, word1.orange.word2 (requires at least one word after orange)
# lazy.#       → Matches: lazy, lazy.anything, lazy.anything.here (lazy + 0+ words)

# Publish messages
channel.basic_publish(
    exchange='exchange_2', 
    routing_key='quick.orange.rabbit', 
    body='Hello World',
    properties=pika.BasicProperties(delivery_mode=2)
)  # → Will match queue_1 (both *.orange.* and *.*.rabbit)

channel.basic_publish(
    exchange='exchange_2', 
    routing_key='lazy.orange.elephant', 
    body='Hello World',
    properties=pika.BasicProperties(delivery_mode=2)
)  # → Will match queue_2 (lazy.#) and queue_1 (*.orange.*)

connection.close()


Fanout Exchange:
Publisher sends 1 message
         ↓
   Fanout Exchange
         ↓
    ┌────┴────┐
    ↓    ↓    ↓
 Queue1 Queue2 Queue3  (Each queue gets its OWN COPY)
    ↓    ↓    ↓
Worker1 Worker2 Worker3  (Each processes independently)


example-1: exchange has multiple queues like SMS, EMAIL if a message published to fanout it will be processed by different queuee  but out put of each queues is different 

example-2: exchange has multiple queues like create order, update inventory, sms, email, create delivary (e-chromers example)


In [ ]:
# Fanout Exchange: Broadcasts to ALL bound queues (ignores routing key)

import pika 

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Declare fanout exchange
channel.exchange_declare(exchange='fanout_exchange', exchange_type='fanout')

# Create multiple queues
channel.queue_declare(queue='sms_queue')
channel.queue_declare(queue='email_queue')
channel.queue_declare(queue='push_notification_queue')

# Bind all queues to the same fanout exchange
channel.queue_bind(exchange='fanout_exchange', queue='sms_queue')
channel.queue_bind(exchange='fanout_exchange', queue='email_queue')
channel.queue_bind(exchange='fanout_exchange', queue='push_notification_queue')

# Publish ONE message - it will be delivered to ALL bound queues
channel.basic_publish(
    exchange='fanout_exchange', 
    routing_key='',  # Routing key is IGNORED in fanout
    body='New order placed: Order #12345'
)

print("Message published - will be received by all 3 queues")

connection.close()


## Headers Exchange

Routes messages based on **message headers** (not routing keys). Uses `x-match` argument:
- `all`: ALL specified headers must match
- `any`: ANY of the specified headers must match


In [ ]:
# Headers Exchange: Routes based on message headers (routing key is IGNORED)

import pika 

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Declare headers exchange
channel.exchange_declare(exchange='headers_exchange', exchange_type='headers')

# Create queues
channel.queue_declare(queue='high_priority_queue')
channel.queue_declare(queue='user_action_queue')

# Bind queues with header matching arguments
# x-match: 'all' means ALL specified headers must match
channel.queue_bind(
    exchange='headers_exchange',
    queue='high_priority_queue',
    arguments={'x-match': 'all', 'priority': 'high', 'type': 'critical'}
)

channel.queue_bind(
    exchange='headers_exchange',
    queue='user_action_queue',
    arguments={'x-match': 'any', 'action': 'create', 'action': 'update'}  # Matches if action is create OR update
)

# Publish message with headers (routing_key is ignored)
channel.basic_publish(
    exchange='headers_exchange',
    routing_key='',  # IGNORED in headers exchange
    body='Critical system alert',
    properties=pika.BasicProperties(
        headers={'priority': 'high', 'type': 'critical'}  # These headers determine routing
    )
)

print("Message published with headers")

connection.close()


## Consumer/Subscriber Implementation

### Basic Consumer Pattern


In [ ]:
# Basic Consumer - Receiving messages from queue

import pika
import time

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Declare queue (must exist before consuming)
channel.queue_declare(queue='task_queue', durable=True)

def callback(ch, method, properties, body):
    """Callback function called when message is received"""
    print(f" [x] Received: {body.decode()}")
    
    # Simulate work
    time.sleep(1)
    
    # Acknowledge message (tell RabbitMQ we processed it)
    ch.basic_ack(delivery_tag=method.delivery_tag)
    print(" [✓] Message processed")

# Fair dispatch: Only give one message to worker at a time
channel.basic_qos(prefetch_count=1)

# Set up consumer
channel.basic_consume(
    queue='task_queue',
    on_message_callback=callback,
    auto_ack=False  # Manual acknowledgment
)

print(" [*] Waiting for messages. To exit press CTRL+C")
channel.start_consuming()


## Complete Example: Logging System (Direct Exchange)

### Real-world use case: Route log messages to different queues based on log level


In [ ]:
# PRODUCER: Logging System Producer (Direct Exchange)

import pika
import json
from datetime import datetime

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Declare direct exchange
exchange_name = 'logs_exchange'
channel.exchange_declare(
    exchange=exchange_name,
    exchange_type='direct',
    durable=True  # Survives server restart
)

# Declare queues (durable = survives server restart)
channel.queue_declare(queue='error_logs_queue', durable=True)
channel.queue_declare(queue='warning_logs_queue', durable=True)
channel.queue_declare(queue='info_logs_queue', durable=True)

# Bind queues to exchange with routing keys
channel.queue_bind(
    exchange=exchange_name,
    queue='error_logs_queue',
    routing_key='error'
)

channel.queue_bind(
    exchange=exchange_name,
    queue='warning_logs_queue',
    routing_key='warning'
)

channel.queue_bind(
    exchange=exchange_name,
    queue='info_logs_queue',
    routing_key='info'
)

# Send log messages
logs = [
    {'level': 'error', 'message': 'Database connection failed'},
    {'level': 'warning', 'message': 'High memory usage detected'},
    {'level': 'info', 'message': 'User logged in successfully'},
    {'level': 'error', 'message': 'Payment processing failed'},
    {'level': 'info', 'message': 'API request processed'}
]

for log in logs:
    message = {
        'timestamp': datetime.now().isoformat(),
        'level': log['level'],
        'message': log['message']
    }
    
    channel.basic_publish(
        exchange=exchange_name,
        routing_key=log['level'],  # Routing key = log level
        body=json.dumps(message),
        properties=pika.BasicProperties(
            delivery_mode=2,  # Make message persistent
            content_type='application/json'
        )
    )
    print(f" [x] Sent {log['level']}: {log['message']}")

connection.close()
print("All logs sent!")


In [ ]:
# CONSUMER: Error Logs Consumer (Direct Exchange)

import pika
import json

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

exchange_name = 'logs_exchange'

# Declare exchange (must match producer)
channel.exchange_declare(
    exchange=exchange_name,
    exchange_type='direct',
    durable=True
)

# Declare queue
channel.queue_declare(queue='error_logs_queue', durable=True)

# Bind queue to exchange
channel.queue_bind(
    exchange=exchange_name,
    queue='error_logs_queue',
    routing_key='error'  # Only receive 'error' level logs
)

def process_error_log(ch, method, properties, body):
    """Process error log messages"""
    log = json.loads(body)
    print(f" [ERROR] {log['timestamp']}: {log['message']}")
    
    # Acknowledge message
    ch.basic_ack(delivery_tag=method.delivery_tag)

channel.basic_consume(
    queue='error_logs_queue',
    on_message_callback=process_error_log,
    auto_ack=False
)

print(" [*] Waiting for ERROR logs. To exit press CTRL+C")
try:
    channel.start_consuming()
except KeyboardInterrupt:
    channel.stop_consuming()
    connection.close()
    print("Consumer stopped")


## Complete Example: E-commerce Order Processing (Fanout Exchange)

### Use case: When order is placed, notify multiple services simultaneously


In [ ]:
# PRODUCER: Order Service (Fanout Exchange)

import pika
import json
from datetime import datetime

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

exchange_name = 'order_exchange'

# Declare fanout exchange
channel.exchange_declare(
    exchange=exchange_name,
    exchange_type='fanout',
    durable=True
)

# Simulate order placement
order = {
    'order_id': 'ORD-12345',
    'user_id': 'USER-789',
    'items': [
        {'product_id': 'PROD-1', 'quantity': 2, 'price': 29.99},
        {'product_id': 'PROD-2', 'quantity': 1, 'price': 49.99}
    ],
    'total_amount': 109.98,
    'timestamp': datetime.now().isoformat()
}

# Publish to fanout exchange - will be delivered to ALL bound queues
channel.basic_publish(
    exchange=exchange_name,
    routing_key='',  # Ignored in fanout
    body=json.dumps(order),
    properties=pika.BasicProperties(
        delivery_mode=2,
        content_type='application/json'
    )
)

print(f" [x] Order placed: {order['order_id']}")
print("   → Notified: Inventory, Email, SMS, Delivery services")

connection.close()


In [ ]:
# CONSUMER: Inventory Service (Fanout Exchange)

import pika
import json

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

exchange_name = 'order_exchange'

channel.exchange_declare(
    exchange=exchange_name,
    exchange_type='fanout',
    durable=True
)

# Create exclusive queue for this consumer
result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue

# Bind to exchange
channel.queue_bind(exchange=exchange_name, queue=queue_name)

def process_order(ch, method, properties, body):
    """Process order in inventory service"""
    order = json.loads(body)
    print(f" [Inventory] Order {order['order_id']}: Updating stock levels...")
    
    for item in order['items']:
        print(f"   → Decrease stock: {item['product_id']} by {item['quantity']}")
    
    print(f" [Inventory] Stock updated for order {order['order_id']}")
    ch.basic_ack(delivery_tag=method.delivery_tag)

channel.basic_consume(
    queue=queue_name,
    on_message_callback=process_order,
    auto_ack=False
)

print(" [Inventory Service] Waiting for orders. To exit press CTRL+C")
try:
    channel.start_consuming()
except KeyboardInterrupt:
    channel.stop_consuming()
    connection.close()
    print("Consumer stopped")


## Message Acknowledgment (ACK/NACK)

### Types of Acknowledgment:
1. **Auto ACK**: Automatically acknowledge on delivery (risky - message lost if consumer crashes)
2. **Manual ACK**: Explicitly acknowledge after processing (recommended)
3. **NACK**: Negative acknowledgment - reject message (can requeue or discard)


In [ ]:
# Message Acknowledgment Examples

import pika
import time

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.queue_declare(queue='ack_example_queue', durable=True)

def process_with_ack(ch, method, properties, body):
    """Example with manual acknowledgment"""
    try:
        print(f" [x] Received: {body.decode()}")
        
        # Simulate processing
        time.sleep(2)
        
        # If processing succeeds, ACK
        ch.basic_ack(delivery_tag=method.delivery_tag)
        print(" [✓] Processed successfully")
        
    except Exception as e:
        print(f" [✗] Error: {e}")
        
        # Reject and requeue on failure
        ch.basic_nack(
            delivery_tag=method.delivery_tag,
            requeue=True  # Put message back in queue for retry
        )
        # OR discard message:
        # ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)

def process_with_auto_ack(ch, method, properties, body):
    """Example with auto acknowledgment (NOT recommended for production)"""
    print(f" [x] Received: {body.decode()}")
    # No need to ACK - auto_ack=True handles it

# Manual ACK (Recommended)
channel.basic_consume(
    queue='ack_example_queue',
    on_message_callback=process_with_ack,
    auto_ack=False  # Manual acknowledgment
)

# Auto ACK (Not recommended - loses messages if consumer crashes)
# channel.basic_consume(
#     queue='ack_example_queue',
#     on_message_callback=process_with_auto_ack,
#     auto_ack=True  # Auto acknowledgment
# )

print(" [*] Waiting for messages. To exit press CTRL+C")
try:
    channel.start_consuming()
except KeyboardInterrupt:
    channel.stop_consuming()
    connection.close()


## Quality of Service (QoS) / Prefetch Count

### Purpose: Control how many messages a consumer receives before acknowledging

- **prefetch_count=1**: Give only 1 unacknowledged message at a time (fair dispatch)
- **prefetch_count=N**: Give N unacknowledged messages (can cause load imbalance)


In [ ]:
# QoS / Prefetch Count Example

import pika
import time

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.queue_declare(queue='qos_example_queue', durable=True)

def process_message(ch, method, properties, body):
    """Process message"""
    print(f" [Worker] Processing: {body.decode()}")
    time.sleep(3)  # Simulate work
    ch.basic_ack(delivery_tag=method.delivery_tag)
    print(f" [Worker] Completed: {body.decode()}")

# Fair dispatch: Only give one message to a worker at a time
# This ensures messages are distributed evenly among multiple workers
channel.basic_qos(prefetch_count=1)

# If prefetch_count=10, worker gets 10 messages before acknowledging
# Faster worker might get all messages, slower worker gets none
# channel.basic_qos(prefetch_count=10)

channel.basic_consume(
    queue='qos_example_queue',
    on_message_callback=process_message,
    auto_ack=False
)

print(" [*] Waiting for messages. To exit press CTRL+C")
try:
    channel.start_consuming()
except KeyboardInterrupt:
    channel.stop_consuming()
    connection.close()


## Durable Queues and Exchanges

### Persistence: Messages survive server restart
- **Durable Queue**: Queue exists even after RabbitMQ restart
- **Durable Exchange**: Exchange exists even after RabbitMQ restart
- **Persistent Message**: Message survives server restart (delivery_mode=2)


In [ ]:
# Durable Queue and Exchange Example

import pika

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Durable Exchange (survives server restart)
channel.exchange_declare(
    exchange='durable_exchange',
    exchange_type='direct',
    durable=True  # Exchange survives RabbitMQ restart
)

# Durable Queue (survives server restart)
channel.queue_declare(
    queue='durable_queue',
    durable=True  # Queue survives RabbitMQ restart
)

channel.queue_bind(
    exchange='durable_exchange',
    queue='durable_queue',
    routing_key='task'
)

# Persistent Message (survives server restart)
channel.basic_publish(
    exchange='durable_exchange',
    routing_key='task',
    body='Important message that must not be lost',
    properties=pika.BasicProperties(
        delivery_mode=2,  # 2 = persistent, 1 = non-persistent
        content_type='text/plain'
    )
)

print("Message published with durability guarantees")
connection.close()

# Note: ALL three must be durable for message to survive restart:
# 1. Durable Exchange ✓
# 2. Durable Queue ✓
# 3. Persistent Message (delivery_mode=2) ✓


## Message Properties

### Common Properties:
- **delivery_mode**: 1 (non-persistent) or 2 (persistent)
- **content_type**: MIME type (e.g., 'application/json', 'text/plain')
- **correlation_id**: For request/response patterns
- **reply_to**: Queue name for responses
- **headers**: Custom headers for routing
- **priority**: Message priority (0-255)
- **expiration**: Message TTL in milliseconds


In [ ]:
# Message Properties Example

import pika
import json
import uuid

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.queue_declare(queue='properties_example_queue')

message = {
    'user_id': '123',
    'action': 'purchase',
    'amount': 99.99
}

correlation_id = str(uuid.uuid4())

# Publish with various properties
channel.basic_publish(
    exchange='',
    routing_key='properties_example_queue',
    body=json.dumps(message),
    properties=pika.BasicProperties(
        delivery_mode=2,  # Persistent
        content_type='application/json',
        correlation_id=correlation_id,  # For request/response tracking
        reply_to='response_queue',  # Queue to send response to
        priority=5,  # Message priority (0-255)
        headers={
            'user_type': 'premium',
            'source': 'web_app'
        },
        expiration='60000'  # Message expires after 60 seconds
    )
)

print(f"Message published with correlation_id: {correlation_id}")
connection.close()


## Dead Letter Queue (DLQ)

### Purpose: Handle messages that couldn't be processed
- Messages are sent to DLQ when:
  - Message is rejected with requeue=False
  - Message TTL expires
  - Queue length limit exceeded
  - Message is rejected too many times


In [ ]:
# Dead Letter Queue Example

import pika

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

# Create dead letter exchange and queue
dlx_name = 'dlx'
dlq_name = 'dlq'

channel.exchange_declare(exchange=dlx_name, exchange_type='direct', durable=True)
channel.queue_declare(queue=dlq_name, durable=True)
channel.queue_bind(exchange=dlx_name, queue=dlq_name, routing_key='failed')

# Create main queue with dead letter configuration
channel.queue_declare(
    queue='main_queue',
    durable=True,
    arguments={
        'x-dead-letter-exchange': dlx_name,  # DLX name
        'x-dead-letter-routing-key': 'failed'  # Routing key for DLQ
    }
)

# Producer: Send message to main queue
channel.basic_publish(
    exchange='',
    routing_key='main_queue',
    body='This message might fail',
    properties=pika.BasicProperties(delivery_mode=2)
)

print("Message sent to main_queue")

# Consumer: If processing fails, message goes to DLQ
def process_message(ch, method, properties, body):
    """Process message - if fails, goes to DLQ"""
    try:
        # Simulate processing
        if body.decode() == 'fail':
            raise Exception("Processing failed")
        
        print(f" [x] Processed: {body.decode()}")
        ch.basic_ack(delivery_tag=method.delivery_tag)
        
    except Exception as e:
        print(f" [✗] Error: {e}")
        # Reject without requeue - goes to DLQ
        ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)

channel.basic_consume(
    queue='main_queue',
    on_message_callback=process_message,
    auto_ack=False
)

print(" [*] Consumer started. Messages that fail will go to DLQ")
# channel.start_consuming()  # Uncomment to start consuming

connection.close()


## Connection Resilience & Error Handling

### Best Practices:
- Retry connection on failure
- Handle connection errors gracefully
- Reconnect automatically
- Close connections properly


In [ ]:
# Connection Resilience Example

import pika
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class RabbitMQConnection:
    """RabbitMQ connection with retry logic"""
    
    def __init__(self, host='localhost', port=5672, max_retries=5, retry_delay=5):
        self.host = host
        self.port = port
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.connection = None
        self.channel = None
    
    def connect(self):
        """Connect with retry logic"""
        for attempt in range(self.max_retries):
            try:
                self.connection = pika.BlockingConnection(
                    pika.ConnectionParameters(
                        host=self.host,
                        port=self.port,
                        connection_attempts=3,
                        retry_delay=2,
                        heartbeat=600,
                        blocked_connection_timeout=300
                    )
                )
                self.channel = self.connection.channel()
                logger.info(f"Connected to RabbitMQ (attempt {attempt + 1})")
                return True
                
            except pika.exceptions.AMQPConnectionError as e:
                logger.error(f"Connection attempt {attempt + 1} failed: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(self.retry_delay)
                else:
                    logger.error("Max retries reached. Connection failed.")
                    return False
            except Exception as e:
                logger.error(f"Unexpected error: {e}")
                return False
        
        return False
    
    def reconnect(self):
        """Reconnect if connection is closed"""
        if self.connection is None or self.connection.is_closed:
            logger.info("Connection closed. Reconnecting...")
            self.connect()
    
    def is_connected(self):
        """Check if connection is alive"""
        return self.connection and not self.connection.is_closed
    
    def close(self):
        """Close connection properly"""
        try:
            if self.channel and not self.channel.is_closed:
                self.channel.close()
            if self.connection and not self.connection.is_closed:
                self.connection.close()
            logger.info("Connection closed successfully")
        except Exception as e:
            logger.error(f"Error closing connection: {e}")

# Usage
rabbit = RabbitMQConnection()
if rabbit.connect():
    try:
        rabbit.channel.queue_declare(queue='resilient_queue')
        rabbit.channel.basic_publish(
            exchange='',
            routing_key='resilient_queue',
            body='Test message'
        )
        print("Message published successfully")
    finally:
        rabbit.close()


In [ ]:
# Topic Exchange - Complete Producer Example

import pika

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.exchange_declare(exchange='topic_logs', exchange_type='topic', durable=True)

# Routing keys and what they match:
# *.orange.* - matches: quick.orange.rabbit, lazy.orange.elephant, etc.
# *.*.rabbit - matches: quick.orange.rabbit, lazy.pink.rabbit, etc.
# lazy.# - matches: lazy.anything.here, lazy, lazy.brown.fox, etc.
# #.orange.# - matches: anything with 'orange' in it

messages = [
    ('quick.orange.rabbit', 'Quick orange rabbit'),
    ('lazy.orange.elephant', 'Lazy orange elephant'),
    ('quick.brown.fox', 'Quick brown fox'),
    ('lazy.pink.rabbit', 'Lazy pink rabbit'),
    ('orange', 'Just orange'),
    ('lazy.brown.fox', 'Lazy brown fox'),
]

for routing_key, message in messages:
    channel.basic_publish(
        exchange='topic_logs',
        routing_key=routing_key,
        body=message,
        properties=pika.BasicProperties(delivery_mode=2)
    )
    print(f" [x] Sent {routing_key}: {message}")

connection.close()


In [ ]:
# Topic Exchange - Consumer Example

import pika
import sys

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

channel.exchange_declare(exchange='topic_logs', exchange_type='topic', durable=True)

# Create temporary exclusive queue
result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue

# Get binding keys from command line arguments
binding_keys = sys.argv[1:] if len(sys.argv) > 1 else ['#']

# Bind queue with pattern(s)
for binding_key in binding_keys:
    channel.queue_bind(
        exchange='topic_logs',
        queue=queue_name,
        routing_key=binding_key
    )
    print(f" [*] Listening for messages matching: {binding_key}")

def callback(ch, method, properties, body):
    print(f" [x] {method.routing_key}: {body.decode()}")
    ch.basic_ack(delivery_tag=method.delivery_tag)

channel.basic_consume(
    queue=queue_name,
    on_message_callback=callback,
    auto_ack=False
)

print(" [*] Waiting for messages. To exit press CTRL+C")
try:
    channel.start_consuming()
except KeyboardInterrupt:
    channel.stop_consuming()
    connection.close()
    print("Consumer stopped")

# Run examples:
# python consumer.py "*.orange.*"          # Matches: quick.orange.rabbit, lazy.orange.elephant
# python consumer.py "*.*.rabbit"           # Matches: quick.orange.rabbit, lazy.pink.rabbit
# python consumer.py "lazy.#"               # Matches: lazy, lazy.orange.elephant, lazy.brown.fox
# python consumer.py "#.orange.#"           # Matches: orange, quick.orange.rabbit, orange.elephant
# python consumer.py "*.orange.*" "lazy.#" # Multiple patterns
# python consumer.py "#"                    # Matches everything


## Topic Exchange Pattern Examples

### Pattern Matching Rules:

| Pattern | Matches | Does NOT Match |
|---------|---------|---------------|
| `*.orange.*` | `quick.orange.rabbit`<br>`lazy.orange.elephant` | `orange`<br>`quick.orange.rabbit.fast` |
| `*.*.rabbit` | `quick.orange.rabbit`<br>`lazy.pink.rabbit` | `rabbit`<br>`quick.rabbit` |
| `lazy.#` | `lazy`<br>`lazy.orange.elephant`<br>`lazy.brown.fox.jumps` | `quick.lazy` |
| `#.orange.#` | `orange`<br>`quick.orange.rabbit`<br>`orange.elephant` | - |
| `#` | Everything | - |

### Key Differences:
- `*.orange.*` requires exactly 3 words, orange in middle
- `#.orange.#` matches orange anywhere (0+ words before/after)
- `#.orange.*` requires at least one word after orange
- `lazy.*` matches exactly 2 words starting with lazy
- `lazy.#` matches lazy + 0 or more words


## Best Practices Summary

### 1. **Always declare queues/exchanges** (idempotent - safe to call multiple times)
```python
channel.queue_declare(queue='my_queue', durable=True)
channel.exchange_declare(exchange='my_exchange', exchange_type='direct', durable=True)
```

### 2. **Use durable queues/exchanges** for production
```python
durable=True  # Survives server restart
```

### 3. **Use persistent messages** for important data
```python
delivery_mode=2  # Message survives server restart
```

### 4. **Use manual acknowledgment** for reliability
```python
auto_ack=False  # Manual ack after processing
ch.basic_ack(delivery_tag=method.delivery_tag)
```

### 5. **Set prefetch_count** for fair dispatch
```python
channel.basic_qos(prefetch_count=1)  # One message at a time per worker
```

### 6. **Handle connection errors** with retry logic
```python
# Implement reconnection logic
```

### 7. **Use appropriate exchange types**
- **Direct**: Exact routing key match (e.g., log levels)
- **Topic**: Pattern-based routing (e.g., order.status.*)
- **Fanout**: Broadcast to all (e.g., notifications)
- **Headers**: Header-based routing (e.g., message metadata)

### 8. **Close connections properly**
```python
try:
    # ... work ...
finally:
    connection.close()
```

### 9. **Use JSON for structured data**
```python
body=json.dumps(data)
content_type='application/json'
```

### 10. **Implement Dead Letter Queues** for error handling
```python
arguments={'x-dead-letter-exchange': 'dlx'}
```


## Quick Reference: Exchange Types Comparison

| Exchange Type | Routing Key | Binding Key | Use Case |
|--------------|-------------|-------------|----------|
| **Direct** | Exact match required | Exact match required | Log levels (error, warning, info) |
| **Topic** | Pattern matching | Wildcards (*, #) | Order status (order.*.success) |
| **Fanout** | Ignored | Ignored | Notifications (SMS, Email, Push) |
| **Headers** | Ignored | Header matching | Complex routing based on metadata |

### Message Flow Examples:

**Direct Exchange:**
```
Publisher → Exchange (direct) → [routing_key='error'] → Queue (binding_key='error')
```

**Topic Exchange:**
```
Publisher → Exchange (topic) → [routing_key='order.payment.success'] 
→ Queue (binding_key='order.*.success') ✓ MATCHES
```

**Fanout Exchange:**
```
Publisher → Exchange (fanout) → ALL bound queues (routing_key ignored)
```

**Headers Exchange:**
```
Publisher → Exchange (headers) → [headers={'priority':'high'}] 
→ Queue (x-match='all', priority='high') ✓ MATCHES
```
